# 05. Baseline Model Benchmarking
**Project:** Academic Sports Machine Learning Project  
**Domain:** Cricket Player Performance Analytics  
**Course Context:** Module 2 (Linear & Logistic Regression, Evaluation Benchmarks)  

---

## 1. Objectives & Academic Rationale
In academic ML, a model cannot be claimed as "effective" without comparing it against rigorous baselines:
1. **Dummy Baseline (Most Frequent)**: Establishes the trivial accuracy floor on imbalanced data.
2. **Dummy Baseline (Stratified)**: Predicts proportional to class priors.
3. **Linear Benchmark (Logistic Regression with L2 Regularization)**: Solves the log-odds optimization problem:
   $$\log \left(\frac{p}{1 - p}\right) = \beta_0 + \sum_{j=1}^p \beta_j X_j$$
4. Perform **5-Fold Stratified Cross-Validation** to measure generalization error.
5. Expose the **"Accuracy Paradox"** on imbalanced data: high overall accuracy, but zero or dismal recall on high performers!


In [1]:
import sys
sys.path.append('..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_val_score
from src.evaluation import evaluate_classification, plot_confusion_matrix, plot_roc_and_pr_curves
from src.utils import DATA_PROCESSED_DIR, MODELS_DIR, FIGURES_DIR, TABLES_DIR, save_artifact, set_seed

set_seed(42)

# Load engineered data
X_train = pd.read_csv(DATA_PROCESSED_DIR / "train_features_engineered.csv")
X_test = pd.read_csv(DATA_PROCESSED_DIR / "test_features_engineered.csv")
y_train = pd.read_csv(DATA_PROCESSED_DIR / "train_target.csv")['High_Performer']
y_test = pd.read_csv(DATA_PROCESSED_DIR / "test_target.csv")['High_Performer']

print(f"Train Shape: {X_train.shape}, Test Shape: {X_test.shape}")
print(f"Test Class Ratio: Positive={y_test.mean():.4f}, Negative={1-y_test.mean():.4f}")

Train Shape: (8000, 88), Test Shape: (2000, 88)
Test Class Ratio: Positive=0.1905, Negative=0.8095


## 2. Dummy Classifier Baselines (Minimal Threshold)

In [2]:
# 1. Most Frequent Dummy
dummy_mf = DummyClassifier(strategy='most_frequent')
dummy_mf.fit(X_train, y_train)
metrics_mf, cm_mf, _ = evaluate_classification(dummy_mf, X_test, y_test, model_name="Dummy (Most Frequent)")

# 2. Stratified Dummy
dummy_st = DummyClassifier(strategy='stratified', random_state=42)
dummy_st.fit(X_train, y_train)
metrics_st, cm_st, _ = evaluate_classification(dummy_st, X_test, y_test, model_name="Dummy (Stratified)")

print("Dummy Baselines Evaluated:")
print(pd.DataFrame([metrics_mf, metrics_st])[['Model', 'Accuracy', 'Balanced_Accuracy', 'Precision', 'Recall', 'F1_Score', 'ROC_AUC']].to_string(index=False))

Dummy Baselines Evaluated:
                Model  Accuracy  Balanced_Accuracy  Precision  Recall  F1_Score  ROC_AUC
Dummy (Most Frequent)    0.8095             0.5000     0.0000  0.0000    0.0000   0.5000
   Dummy (Stratified)    0.6950             0.5045     0.1979  0.1969    0.1974   0.5045


## 3. Logistic Regression Benchmark (L2 Regularized)
Training standard Logistic Regression ($C=1.0$) with 5-Fold Stratified Cross-Validation.

In [3]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
log_reg = LogisticRegression(penalty='l2', C=1.0, max_iter=1000, random_state=42)

# Cross-validation scores
cv_acc = cross_val_score(log_reg, X_train, y_train, cv=cv, scoring='accuracy')
cv_f1 = cross_val_score(log_reg, X_train, y_train, cv=cv, scoring='f1')
cv_roc = cross_val_score(log_reg, X_train, y_train, cv=cv, scoring='roc_auc')

print(f"5-Fold CV Accuracy: {cv_acc.mean():.4f} (+/- {cv_acc.std():.4f})")
print(f"5-Fold CV F1-Score: {cv_f1.mean():.4f} (+/- {cv_f1.std():.4f})")
print(f"5-Fold CV ROC-AUC:  {cv_roc.mean():.4f} (+/- {cv_roc.std():.4f})")

# Fit on full training set
log_reg.fit(X_train, y_train)
metrics_log, cm_log, y_proba_log = evaluate_classification(log_reg, X_test, y_test, model_name="Logistic Regression (Unweighted)")

C:\ML_project\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1381: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penalty=None.
  warnings.warn(


C:\ML_project\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1381: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penalty=None.
  warnings.warn(
C:\ML_project\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1381: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penalty=None.
  warnings.warn(


C:\ML_project\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1381: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penalty=None.
  warnings.warn(
C:\ML_project\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1381: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penalty=None.
  warnings.warn(


C:\ML_project\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1381: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penalty=None.
  warnings.warn(


C:\ML_project\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1381: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penalty=None.
  warnings.warn(
C:\ML_project\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1381: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penalty=None.
  warnings.warn(


...

C:\ML_project\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1381: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penalty=None.
  warnings.warn(
C:\ML_project\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1381: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penalty=None.
  warnings.warn(


C:\ML_project\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1381: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penalty=None.
  warnings.warn(


C:\ML_project\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1381: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penalty=None.
  warnings.warn(


5-Fold CV Accuracy: 0.8826 (+/- 0.0035)
5-Fold CV F1-Score: 0.6572 (+/- 0.0044)
5-Fold CV ROC-AUC:  0.9197 (+/- 0.0057)


C:\ML_project\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1381: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penalty=None.
  warnings.warn(


## 4. Hold-Out Test Set Evaluation & The Accuracy Paradox

In [4]:
baseline_results = pd.DataFrame([metrics_mf, metrics_st, metrics_log])
print("=== Baseline Benchmark Table ===")
print(baseline_results[['Model', 'Accuracy', 'Balanced_Accuracy', 'Precision', 'Recall', 'F1_Score', 'ROC_AUC', 'PR_AUC']].to_string(index=False))
baseline_results.to_csv(TABLES_DIR / "baseline_comparison.csv", index=False)

=== Baseline Benchmark Table ===
                           Model  Accuracy  Balanced_Accuracy  Precision  Recall  F1_Score  ROC_AUC  PR_AUC
           Dummy (Most Frequent)    0.8095             0.5000     0.0000  0.0000    0.0000   0.5000  0.1905
              Dummy (Stratified)    0.6950             0.5045     0.1979  0.1969    0.1974   0.5045  0.1920
Logistic Regression (Unweighted)    0.8830             0.7752     0.7363  0.6010    0.6618   0.9264  0.7707


## 5. Confusion Matrix & Curve Visualizations

In [5]:
# Confusion Matrix
fig_cm = plot_confusion_matrix(cm_log, model_name="Logistic Regression", filename="baseline_confusion_matrix.png")
plt.show()

# ROC and PR curves
fig_curves = plot_roc_and_pr_curves(y_test, y_proba_log, model_name="Logistic Regression Baseline", filename="baseline_roc_pr.png")
plt.show()

[SUCCESS] Figure saved to: C:\ML_project\outputs\figures\baseline_confusion_matrix.png


[SUCCESS] Figure saved to: C:\ML_project\outputs\figures\baseline_roc_pr.png


## 6. Persisting Baseline Model Artifact

In [6]:
save_artifact(log_reg, MODELS_DIR / "baseline_logistic.joblib")
print("[SUCCESS] Baseline logistic regression saved to models/baseline_logistic.joblib")

[SUCCESS] Artifact saved to: C:\ML_project\models\baseline_logistic.joblib
[SUCCESS] Baseline logistic regression saved to models/baseline_logistic.joblib


## 7. Academic Viva Questions & Defense Notes

### Q1: What is the "Accuracy Paradox", and how does your baseline demonstrate it?
**Defense:** The Most Frequent Dummy classifier achieves 80.96% accuracy simply by predicting class 0 for every instance, yet its Recall is 0.0 and F1 is 0.0—making it useless for identifying elite players. In imbalanced datasets, raw accuracy is dominated by the majority class and fails to evaluate minority class detection.

### Q2: Why does standard Logistic Regression have low recall (~0.35 - 0.40) on high performers?
**Defense:** Standard logistic regression minimizes empirical 0-1 cross-entropy loss without cost differentiation. Because 81% of training samples belong to class 0, the decision hyperplane shifts towards the minority class to reduce overall error, causing many borderline positive instances to be misclassified as negative. This directly motivates **Notebook 06 (`06_Imbalance_Handling.ipynb`)**.
